# Experiment 5 — Language- and Domain-Aware Stacking
**Question:** Can knowing a document's language and domain improve the probability stacking model? The existing 25% stacking blend scored **0.91829 on the public Kaggle leaderboard**. This experiment uses the same 8,000 validation IDs and the same pretrained predictions. 

Two possible explanations are tested: **language effects** (English vs. German) and **domain effects** (news, books, parliamentary debates, etc.). Metadata (`lang`, `domain`) is supplied by the competition; **no test labels and no external data** are used.

**Protocol:** 4 outer stratified folds. On each outer training fold, temperature calibration and meta-model fitting are learned without the holdout; the regularization parameter is selected with 3 inner folds. All candidates are evaluated on identical holdout rows.

In [1]:
from pathlib import Path
import glob, hashlib, json, warnings
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, classification_report, confusion_matrix, log_loss

SEED=42
CLASSES=['human','ai','fusion']
L2I={c:i for i,c in enumerate(CLASSES)}
EXPECTED_FP='97fed7e0aa8163e74359d495ffc4a025'
OUT=Path('/kaggle/working/experiment_5_domain_stacking_outputs') if Path('/kaggle/working').exists() else Path('./experiment_5_domain_stacking_outputs')
OUT.mkdir(parents=True, exist_ok=True)
TFIDF_PATH=None; XLMR_PATH=None; BASELINE_SUBMISSION_PATH=None
print('Output directory:',OUT)


Output directory: /kaggle/working/experiment_5_domain_stacking_outputs


## 1. Data, split, and strict prediction alignment checks

In [2]:
def unique_file(name, explicit=None):
    if explicit is not None:
        p=Path(explicit)
        if not p.is_file(): raise FileNotFoundError(str(p))
        return p
    paths=sorted(set(str(p) for root in ['/kaggle/input','.'] for p in Path(root).glob('**/'+name)))
    if not paths: raise FileNotFoundError(f'{name} missing; add the relevant Kaggle input')
    if len(paths)>1: raise RuntimeError(f'Multiple {name}: {paths}. Set explicit path.')
    return Path(paths[0])

def check_probs(P,n,name):
    P=np.asarray(P,dtype=np.float64)
    if P.shape!=(n,3) or not np.isfinite(P).all() or np.any(P<0) or np.any(P>1+1e-4):
        raise ValueError(f'Invalid probabilities: {name}, shape={P.shape}')
    if not np.allclose(P.sum(axis=1),1,atol=1e-3): raise ValueError(f'Probability sums: {name}')
    P=np.clip(P,1e-12,1.)
    return P/P.sum(axis=1,keepdims=True)

def indexer(from_ids,to_ids):
    a=pd.Index([str(z) for z in from_ids]); b=pd.Index([str(z) for z in to_ids])
    if not a.is_unique or len(a)!=len(b) or set(a)!=set(b): raise ValueError('ID mismatch')
    return a.get_indexer(b)

train_path=unique_file('train.csv'); data_dir=train_path.parent
train=pd.read_csv(train_path,dtype={'id':str})
test=pd.read_csv(data_dir/'test.csv',dtype={'id':str})
template=pd.read_csv(data_dir/'sample_submission.csv',dtype={'id':str})
assert len(test)==len(template)==25000 and train.id.is_unique and test.id.is_unique
assert set(test.id)==set(template.id)
sub=train.sample(40000,random_state=SEED).reset_index(drop=True)
_,val=train_test_split(sub,test_size=.2,random_state=SEED,stratify=sub.label+'_'+sub.lang)
val=val.reset_index(drop=True)
val_ids=val.id.astype(str).tolist(); test_ids=test.id.astype(str).tolist()
y=val.label.map(L2I).to_numpy(dtype=np.int64)
fp=hashlib.md5(','.join(val_ids).encode()).hexdigest()
assert fp==EXPECTED_FP,(fp,EXPECTED_FP)
print('Validation ID fingerprint:',fp)

xp=unique_file('xlmr_probs.npz',XLMR_PATH)
tp=unique_file('tfidf_probs.npz',TFIDF_PATH)
with np.load(xp,allow_pickle=False) as z:
    assert set(['classes','va_ids','test_ids','yva','P_va','P_te']).issubset(z.files)
    assert list(map(str,z['classes']))==CLASSES
    ix=indexer(z['va_ids'],val_ids); it=indexer(z['test_ids'],test_ids)
    X=check_probs(z['P_va'][ix],len(y),'XLMR valid')
    Xt=check_probs(z['P_te'][it],len(test),'XLMR test')
    assert np.array_equal(z['yva'][ix].astype(int),y)
run_cfg=xp.parent/'run_config.json'
if run_cfg.is_file():
    config=json.loads(run_cfg.read_text()); assert config.get('best_epoch')==4
    if config.get('val_id_fingerprint'): assert config['val_id_fingerprint']==fp
else:
    print('WARNING: run_config.json not found; checking archive against recorded score.')
with np.load(tp,allow_pickle=False) as z:
    assert set(['yva','P_va','P_te']).issubset(z.files)
    if 'classes' in z.files: assert list(map(str,z['classes']))==CLASSES
    if 'va_ids' in z.files and 'test_ids' in z.files:
        ix=indexer(z['va_ids'],val_ids); it=indexer(z['test_ids'],test_ids)
        T=check_probs(z['P_va'][ix],len(y),'TFIDF valid')
        Tt=check_probs(z['P_te'][it],len(test),'TFIDF test')
        assert np.array_equal(z['yva'][ix].astype(int),y)
        tf_has_ids=True
    else:
        T=check_probs(z['P_va'],len(y),'TFIDF valid')
        Tt=check_probs(z['P_te'],len(test),'TFIDF test')
        assert np.array_equal(z['yva'].astype(int),y)
        tf_has_ids=False

if not tf_has_ids:
    bs=unique_file('sub_tfidf_baseline.csv',BASELINE_SUBMISSION_PATH)
    df=pd.read_csv(bs,dtype={'id':str})
    assert df.id.is_unique and set(df.id)==set(test_ids)
    assert np.array_equal(df.set_index('id').loc[test_ids,'label'].to_numpy(),np.array(CLASSES)[Tt.argmax(1)]), 'TF-IDF test row order mismatch'
    print('TF-IDF test order confirmed by baseline submission IDs')
ref_old=f1_score(y,(.45*X+.55*T).argmax(1),average='macro')
assert abs(ref_old-.9141)<.002 and abs(f1_score(y,X.argmax(1),average='macro')-.9049)<.002
print('Original XLM-R + TF-IDF 45/55:',round(ref_old,5),'| validation:',len(y),'| test:',len(test))

val_meta=val[['lang','domain']].fillna('unknown').astype(str).reset_index(drop=True)
test_meta=test[['lang','domain']].fillna('unknown').astype(str).reset_index(drop=True)
print('Validation languages:',val_meta.lang.value_counts().to_dict())
print('Validation domains:',val_meta.domain.value_counts().to_dict())


Validation ID fingerprint: 97fed7e0aa8163e74359d495ffc4a025
TF-IDF test order confirmed by baseline submission IDs
Original XLM-R + TF-IDF 45/55: 0.91409 | validation: 8000 | test: 25000
Validation languages: {'en-EN': 6454, 'de-DE': 1546}
Validation domains: {'student_essays': 1579, 'bundestag': 1535, 'cnn_news': 1290, 'blog_authorship_corpus': 1160, 'house_of_commons': 948, 'euro_court_cases': 862, 'arxiv_papers': 427, 'gutenberg': 199}


## 2. Cross-fitted temperature calibration and metadata-augmented features

Reference = temperature-scaled XLM-R and TF-IDF averaged 50/50. The stacker receives six calibrated **log probabilities**. Language/domain versions add categorical features to this six-dimensional vector using a one-hot encoder fit **only on the training fold**. Comparison is made at equal validation IDs; no TF-IDF vectorizers are re-fitted.

In [3]:
EPS=1e-12

def temp_scale(P,t):
    Z=np.log(np.clip(P,EPS,1.0))/float(t)
    Z-=Z.max(axis=1,keepdims=True)
    Q=np.exp(Z)
    return Q/Q.sum(axis=1,keepdims=True)

def fit_temp(P,labels):
    def nll(z):
        Q=temp_scale(P,np.exp(z))
        return -np.log(np.clip(Q[np.arange(len(labels)),labels],EPS,1.)).mean()
    r=minimize_scalar(nll,bounds=(np.log(.05),np.log(20)),method='bounded')
    if not r.success: raise RuntimeError(r.message)
    return float(np.exp(r.x))

def features(Qx,Qt):
    return np.hstack([np.log(np.clip(Qx,EPS,1.)),np.log(np.clip(Qt,EPS,1.))])

def feature_frame(Qx,Qt,metadata):
    F=features(Qx,Qt)
    df=pd.DataFrame(F,columns=[f'x_{k}' for k in CLASSES]+[f't_{k}' for k in CLASSES])
    for name in ['lang','domain']:
        df[name]=metadata[name].to_numpy()
    return df

def make_meta(C, mode):
    if mode=='plain':
        return make_pipeline(StandardScaler(),LogisticRegression(C=float(C),max_iter=1500,random_state=SEED))
    if mode not in ['language','domain','both']:
        raise ValueError(mode)
    categorical={'language':['lang'],'domain':['domain'],'both':['lang','domain']}[mode]
    numerical=[f'x_{k}' for k in CLASSES]+[f't_{k}' for k in CLASSES]
    pre=ColumnTransformer([
        ('numeric',StandardScaler(),numerical),
        ('categorical',OneHotEncoder(handle_unknown='ignore'),categorical)
    ],remainder='drop')
    return make_pipeline(pre,LogisticRegression(C=float(C),max_iter=1500,random_state=SEED))

def train_features(Qx,Qt,idx,mode):
    if mode=='plain':
        return features(Qx,Qt)
    return feature_frame(Qx,Qt,val_meta.iloc[idx])

def test_features(Qx,Qt,mode):
    if mode=='plain':
        return features(Qx,Qt)
    return feature_frame(Qx,Qt,test_meta)

def get_fold_data(fit_idx,eval_idx,mode):
    tx=fit_temp(X[fit_idx],y[fit_idx]); tt=fit_temp(T[fit_idx],y[fit_idx])
    xtrain=temp_scale(X[fit_idx],tx); ttrain=temp_scale(T[fit_idx],tt)
    xeval=temp_scale(X[eval_idx],tx); teval=temp_scale(T[eval_idx],tt)
    return (train_features(xtrain,ttrain,fit_idx,mode),
            train_features(xeval,teval,eval_idx,mode),
            .5*(xeval+teval),tx,tt)

def choose_C(train_idx,mode,grid=(.03,.1,.3,1.,3.)):
    yy=y[train_idx]
    splitter=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED)
    scores={c:[] for c in grid}
    for ii,jj in splitter.split(np.zeros(len(train_idx)),yy):
        indices_fit=train_idx[ii]; indices_val=train_idx[jj]
        Ftr,Fva,_,_,_=get_fold_data(indices_fit,indices_val,mode)
        for c in grid:
            model=make_meta(c,mode).fit(Ftr,y[indices_fit])
            pred=model.predict(Fva)
            scores[c].append(f1_score(y[indices_val],pred,average='macro'))
    winner=max(grid,key=lambda c:(np.mean(scores[c]),-c))
    return winner,{str(c):round(float(np.mean(scores[c])),5) for c in grid}


## 3. Outer cross-fitting — compare the same held-out documents

Candidates include existing **25% plain stacking** and metadata-aware stacking with language, domain, or both. For each metadata model, evaluate a **25% blend** with the reference and its pure predictions. Choosing the highest-scoring method after observing these outcomes can be optimistic; use fold consistency, corrected-vs-broken predictions, and bootstrap differences as diagnostics.

In [4]:
outer=StratifiedKFold(n_splits=4,shuffle=True,random_state=SEED+7)
MODES=['plain','language','domain','both']
KEYS=['reference','plain25']+[f'{m}_{blend}' for m in MODES if m!='plain' for blend in ['25','pure']]
prob_oof={key:np.full((len(y),3),np.nan) for key in KEYS}
fold_results=[]
for fold,(fi,ei) in enumerate(outer.split(np.zeros(len(y)),y),1):
    row={'fold':fold,'fit_rows':len(fi),'heldout_rows':len(ei)}
    reference_saved=None
    for mode in MODES:
        Cbest,inner=choose_C(fi,mode)
        Ftr,Fev,reference,tx,tt=get_fold_data(fi,ei,mode)
        model=make_meta(Cbest,mode).fit(Ftr,y[fi])
        stack=model.predict_proba(Fev)
        assert np.array_equal(model[-1].classes_,np.arange(3))
        if reference_saved is None:
            prob_oof['reference'][ei]=reference
            reference_saved=reference
        else:
            assert np.allclose(reference_saved,reference)
        row[f'{mode}_C']=Cbest
        row[f'{mode}_inner_macro_f1']=max(inner.values())
        if mode=='plain':
            prob_oof['plain25'][ei]=.75*reference+.25*stack
        else:
            prob_oof[f'{mode}_25'][ei]=.75*reference+.25*stack
            prob_oof[f'{mode}_pure'][ei]=stack
    for key in KEYS:
        row[key+'_f1']=f1_score(y[ei],prob_oof[key][ei].argmax(1),average='macro')
    fold_results.append(row)
    print('Fold',fold,'|', {k:round(row[k+'_f1'],5) for k in KEYS})
for key,P in prob_oof.items(): assert np.isfinite(P).all(),key
fold_df=pd.DataFrame(fold_results)
fold_df.to_csv(OUT/'fold_metrics.csv',index=False)
display(fold_df.round(5))

Fold 1 | {'reference': 0.90907, 'plain25': 0.9156, 'language_25': 0.91374, 'language_pure': 0.92009, 'domain_25': 0.91389, 'domain_pure': 0.92191, 'both_25': 0.91389, 'both_pure': 0.92069}
Fold 2 | {'reference': 0.92308, 'plain25': 0.92969, 'language_25': 0.93017, 'language_pure': 0.93526, 'domain_25': 0.92779, 'domain_pure': 0.94022, 'both_25': 0.92779, 'both_pure': 0.94022}
Fold 3 | {'reference': 0.91925, 'plain25': 0.92702, 'language_25': 0.92702, 'language_pure': 0.92878, 'domain_25': 0.92654, 'domain_pure': 0.93135, 'both_25': 0.92717, 'both_pure': 0.9315}
Fold 4 | {'reference': 0.92274, 'plain25': 0.92774, 'language_25': 0.92834, 'language_pure': 0.93373, 'domain_25': 0.92771, 'domain_pure': 0.92991, 'both_25': 0.92771, 'both_pure': 0.9307}


,fold,fit_rows,heldout_rows,plain_C,plain_inner_macro_f1,language_C,language_inner_macro_f1,domain_C,domain_inner_macro_f1,both_C,both_inner_macro_f1,reference_f1,plain25_f1,language_25_f1,language_pure_f1,domain_25_f1,domain_pure_f1,both_25_f1,both_pure_f1
0,1,6000,2000,3.0,0.93029,3.0,0.93193,3.0,0.93127,3.0,0.93108,0.90907,0.91560,0.91374,0.92009,0.91389,0.92191,0.91389,0.92069
1,2,6000,2000,3.0,0.92595,3.0,0.92613,3.0,0.92577,3.0,0.92516,0.92308,0.92969,0.93017,0.93526,0.92779,0.94022,0.92779,0.94022
2,3,6000,2000,3.0,0.92775,3.0,0.92855,3.0,0.92989,3.0,0.93009,0.91925,0.92702,0.92702,0.92878,0.92654,0.93135,0.92717,0.93150
3,4,6000,2000,3.0,0.92718,3.0,0.92812,3.0,0.92922,3.0,0.93002,0.92274,0.92774,0.92834,0.93373,0.92771,0.92991,0.92771,0.93070


## 4. Comparative results and robustness checks

The primary benchmark is **plain25**, the previously submitted 25% stacking blend, rather than the weaker 50/50 calibrated-only reference. Report cross-fitted macro-F1, deltas against plain25, per-class F1, and paired bootstrap intervals. The bootstrap treats documents as sampling units and does not correct for model selection or related-source groups.

In [5]:
baseline_pred=prob_oof['plain25'].argmax(1)
baseline_f1=f1_score(y,baseline_pred,average='macro')
rows=[]
for name,P in prob_oof.items():
    pred=P.argmax(1)
    rows.append({'model':name,'crossfit_macro_f1':f1_score(y,pred,average='macro'),
                'delta_vs_plain25':f1_score(y,pred,average='macro')-baseline_f1,
                'changed_vs_plain25':int((pred!=baseline_pred).sum()),
                'corrected':int(((pred==y)&(baseline_pred!=y)).sum()),
                'broken':int(((pred!=y)&(baseline_pred==y)).sum()),
                **{c+'_f1':f1_score(y,pred,labels=[i],average='macro',zero_division=0) for i,c in enumerate(CLASSES)}})
comp=pd.DataFrame(rows).sort_values('crossfit_macro_f1',ascending=False)
comp.to_csv(OUT/'model_comparison.csv',index=False)
display(comp.round(5))
rng=np.random.default_rng(SEED); B=1000
for key in KEYS:
    if key=='plain25': continue
    yh=prob_oof[key].argmax(1)
    differences=[]
    for j in range(B):
        ix=rng.integers(len(y),size=len(y))
        differences.append(f1_score(y[ix],yh[ix],average='macro')-f1_score(y[ix],baseline_pred[ix],average='macro'))
    ci=np.quantile(differences,[.025,.975])
    print(f'{key}: delta={f1_score(y,yh,average="macro")-baseline_f1:+.5f}, bootstrap CI [{ci[0]:+.5f}, {ci[1]:+.5f}]')
print('Baseline plain25 macro-F1:',round(baseline_f1,5))
print('WARNING: comparing and selecting among several candidates on the same OOF validation may overstate improvement.')

,model,crossfit_macro_f1,delta_vs_plain25,changed_vs_plain25,corrected,broken,human_f1,ai_f1,fusion_f1
5,domain_pure,0.93085,0.00584,101,69,30,0.88769,0.99583,0.90902
7,both_pure,0.93077,0.00577,104,70,32,0.88803,0.99556,0.90872
3,language_pure,0.92945,0.00445,74,50,20,0.88552,0.99597,0.90687
1,plain25,0.92500,0.00000,0,0,0,0.87854,0.99404,0.90243
2,language_25,0.92481,-0.00020,7,3,4,0.87796,0.99418,0.90229
6,both_25,0.92413,-0.00087,34,14,20,0.87738,0.99376,0.90125
4,domain_25,0.92397,-0.00103,33,13,20,0.87708,0.99376,0.90107
0,reference,0.91852,-0.00648,94,19,67,0.87189,0.98897,0.89470


reference: delta=-0.00648, bootstrap CI [-0.00888, -0.00409]
language_25: delta=-0.00020, bootstrap CI [-0.00098, +0.00057]
language_pure: delta=+0.00445, bootstrap CI [+0.00195, +0.00714]
domain_25: delta=-0.00103, bootstrap CI [-0.00261, +0.00051]
domain_pure: delta=+0.00584, bootstrap CI [+0.00300, +0.00890]
both_25: delta=-0.00087, bootstrap CI [-0.00250, +0.00095]
both_pure: delta=+0.00577, bootstrap CI [+0.00305, +0.00866]
Baseline plain25 macro-F1: 0.925


## 5. Fit full-validation meta-models and create candidate submissions

Temperatures are fitted on all 8,000 labeled validation texts for final test inference. C is chosen by inner 3-fold CV.

In [6]:
all_idx=np.arange(len(y))
tx=fit_temp(X,y); tt=fit_temp(T,y)
Qx=temp_scale(X,tx); Qt=temp_scale(T,tt)
Qxt=temp_scale(Xt,tx); Qtt=temp_scale(Tt,tt)
reference_test=.5*(Qxt+Qtt)
predictions={'sub_calibrated_5050_reference.csv':reference_test}
params={'experiment':'metadata-aware regularized stacking','validation_fp':fp,'temperatures':{'XLMR':tx,'TFIDF':tt},'models':{}}
for mode in MODES:
    best_c,inner=choose_C(all_idx,mode)
    model=make_meta(best_c,mode).fit(train_features(Qx,Qt,all_idx,mode),y)
    pred=model.predict_proba(test_features(Qxt,Qtt,mode))
    assert np.array_equal(model[-1].classes_,np.arange(3))
    params['models'][mode]={'C':best_c,'inner_scores':inner}
    if mode=='plain':
        predictions['sub_stacking_25_reference.csv']=.75*reference_test+.25*pred
    else:
        predictions[f'sub_{mode}_stack25.csv']=.75*reference_test+.25*pred
        predictions[f'sub_{mode}_stack_pure.csv']=pred

def save_submission(name,P):
    df=pd.DataFrame({'id':test_ids,'label':np.array(CLASSES)[P.argmax(1)]})
    df=df.set_index('id').loc[template.id].reset_index()
    assert len(df)==25000 and df.id.is_unique and list(df.columns)==['id','label']
    assert df.label.isin(CLASSES).all()
    df.to_csv(OUT/name,index=False)
    return df
plain_sub=None
for name,P in predictions.items():
    df=save_submission(name,P)
    if name=='sub_stacking_25_reference.csv': plain_sub=df.label.to_numpy()
for name,P in predictions.items():
    df=pd.read_csv(OUT/name,dtype={'id':str})
    print(name,'| changes vs plain25:',int((df.label.to_numpy()!=plain_sub).sum()))
np.savez_compressed(OUT/'metadata_stack_oof.npz',va_ids=np.array(val_ids),test_ids=np.array(test_ids),yva=y,**{f'{k}_oof':P for k,P in prob_oof.items()})
(OUT/'config.json').write_text(json.dumps(params,indent=2))
print('Outputs:')
for p in sorted(OUT.iterdir()): print('-',p.name)

sub_calibrated_5050_reference.csv | changes vs plain25: 315
sub_stacking_25_reference.csv | changes vs plain25: 0
sub_language_stack25.csv | changes vs plain25: 22
sub_language_stack_pure.csv | changes vs plain25: 239
sub_domain_stack25.csv | changes vs plain25: 73
sub_domain_stack_pure.csv | changes vs plain25: 299
sub_both_stack25.csv | changes vs plain25: 72
sub_both_stack_pure.csv | changes vs plain25: 289
Outputs:
- config.json
- fold_metrics.csv
- metadata_stack_oof.npz
- model_comparison.csv
- sub_both_stack25.csv
- sub_both_stack_pure.csv
- sub_calibrated_5050_reference.csv
- sub_domain_stack25.csv
- sub_domain_stack_pure.csv
- sub_language_stack25.csv
- sub_language_stack_pure.csv
- sub_stacking_25_reference.csv


## Interpretation

This experiment evaluates whether known language/domain metadata helps the meta-classifier interpret disagreement between TF-IDF and XLM-R. The relevant comparison is to **plain25**, whose Kaggle public score was **0.91829**. Metadata-aware changes may help in some domains and hurt in others because train and test have different domain proportions. The code reports cross-fitted performance, fold-by-fold consistency, class-level changes, and changed test predictions without claiming an improvement before the actual results are available.